<a href="https://colab.research.google.com/github/ilkecerci/cross-domain-turkish-sentiment-analysis/blob/main/notebooks/03_external_validation_dataset_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# load existing splits
train_df = pd.read_csv('/content/drive/MyDrive/Bitirme Projesi/train_natural.csv')
val_df = pd.read_csv('/content/drive/MyDrive/Bitirme Projesi/val_natural.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Bitirme Projesi/test_natural.csv')

# gather all unique texts from the existing 4 domains
existing_texts = set(train_df['text'].dropna().str.lower().str.strip()) | \
                 set(val_df['text'].dropna().str.lower().str.strip()) | \
                 set(test_df['text'].dropna().str.lower().str.strip())

# load external validation dataset
ext_dataset_path = '/content/drive/MyDrive/Bitirme Projesi/turkish_movie_sentiment_dataset.csv'
ext_df = pd.read_csv(ext_dataset_path)

print(f"initial rows in external dataset: {len(ext_df)}")

# Fix for TypeError: Convert point column to float
# Replace commas with dots if they exist as strings, then convert to numeric
ext_df['point'] = ext_df['point'].astype(str).str.replace(',', '.').astype(float)

# drop neutral ratings (3.0) and map 0-2 to Negative, 4-5 to Positive
ext_df = ext_df[ext_df['point'] != 3.0]
ext_df['label'] = ext_df['point'].apply(lambda x: 'Positive' if x >= 4.0 else 'Negative')

# clean text for exact matching
ext_df['clean_text'] = ext_df['comment'].astype(str).str.lower().str.strip()

# remove reviews that have conflicting labels
conflicting_texts = ext_df.groupby('clean_text')['label'].nunique()
conflicting_texts = conflicting_texts[conflicting_texts > 1].index
ext_df = ext_df[~ext_df['clean_text'].isin(conflicting_texts)]

# drop duplicates within the external dataset itself
ext_df = ext_df.drop_duplicates(subset=['clean_text'], keep='first')
print(f"rows after dropping duplicates and conflicts: {len(ext_df)}")

# remove overlapping reviews (present in our main datasets)
ext_df_final = ext_df[~ext_df['clean_text'].isin(existing_texts)].copy()
overlap_count = len(ext_df) - len(ext_df_final)

print("\n--- external dataset final summary ---")
print(f"overlapping reviews removed: {overlap_count}")
print(f"final total reviews: {len(ext_df_final)}\n")
print("final class distribution:")
print(ext_df_final['label'].value_counts().to_string())

# save final cleaned dataset
save_path = '/content/drive/MyDrive/Bitirme Projesi/external_validation_final.csv'
ext_df_final[['comment', 'label']].to_csv(save_path, index=False)
print(f"\nsaved final dataset to {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
initial rows in external dataset: 83227
rows after dropping duplicates and conflicts: 76939

--- external dataset final summary ---
overlapping reviews removed: 7317
final total reviews: 69622

final class distribution:
label
Positive    40501
Negative    29121

saved final dataset to /content/drive/MyDrive/Bitirme Projesi/external_validation_final.csv
